# 01 — Data profiling

Row counts, null profiles, grain verification, and the quality findings of §7.

**Rules (§6, §9).** No cleaning happens in a notebook — cleaning here means each
analyst gets different numbers, which is the justification for the warehouse
existing at all. Read `olist_marts` only, via SQLAlchemy; never `raw` or
`staging`.

Owner: lane B2.


In [ ]:
uv pip install sqlalchemy

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [ ]:
engine = create_engine(
    #'bigquery://heroic-vial-505103-a3/olist'
    'bigquery://sctp-dsai-gcp-project/dbt_dev_marts'
    #credentials_path='/path/to/service-account-key.json'
)

# warm up lol.
with engine.connect() as conn:
    df = pd.read_sql("SELECT * FROM dbt_dev_marts.fct_orders LIMIT 10", conn)

display(df)

---

## Star-schema validation — `dbt_dev_marts` rebuilt from `olist_raw`

Checked the marts layer against the raw landing tables (metadata + key/measure
queries only, to keep BigQuery scan cost negligible). **Verdict: the star schema
is built correctly — no structural mistakes.** It is a fact *constellation*
(4 facts) sharing 4 conformed dimensions.

### What passed

| Area | Check | Result |
|---|---|---|
| **Grain** | Every fact row count == raw source row count | `fct_order_items` 112,650 · `fct_orders` 99,441 · `fct_payments` 103,886 all match raw exactly — no fan-out, no drops |
| **Grain** | `fct_reviews` = raw de-duplicated on `review_id` | 99,224 raw → 98,410 unique (814 dupes removed), matches `fct_reviews` row count |
| **Dim keys** | `dim_*` key is unique, non-null, and a subset of the raw natural key | `dim_customer` 96,096 · `dim_product` 32,951 · `dim_seller` 3,095 — all pass, 0 orphans |
| **Critical decision** | `dim_customer` keyed on `customer_unique_id` (96,096), **not** `customer_id` (99,441) | Correct — the §5.2 repeat-customer trap is avoided; `< fct_orders` guard holds |
| **Referential integrity** | Every FK in every fact resolves to its dimension | 0 orphan `customer_key` / `product_key` / `seller_key` / `order_id` across all facts |
| **Conformed date** | `dim_date` contiguous 2016-09-01 → 2018-12-31 (852 days), all fact dates present | 0 dates missing from `dim_date`; `is_complete_month` cuts off at 2018-08-31 (§9) |
| **Measures** | Additive measures reconcile to the cent | `SUM(price)` 13,591,643.70 · `SUM(freight)` 2,251,909.54 · `SUM(payment_value)` 16,008,872.12 — mart == raw |
| **Measures** | `fct_orders` roll-ups == `fct_order_items` aggregation; payment totals == raw | 0 mismatches; `line_gross_value == price + freight_value` everywhere |
| **Unknown member** | `dim_product.product_category_name_english` never null | 0 nulls — 623 mapped to `'unknown'` |
| **Values** | No negative money; `review_score` in 1–5; `order_status` matches raw | all pass |

### Minor observations (design choices, not defects)

1. **`is_repeat_customer` is defined on *delivered* orders, not placed orders.**
   2,801 customers have `delivered_order_count > 1` and are flagged; 2,997 have
   `order_count > 1`. So 196 customers who placed ≥2 orders are **not** flagged
   repeat (≤1 was delivered). Defensible, but RFM / retention analysis must know
   which definition is in force.
2. **`'unknown'` category = 623 = 610 raw NULLs + 13 products in 2 categories
   missing from the translation table** (`pc_gamer`,
   `portateis_cozinha_e_preparadores_de_alimentos`). The Portuguese
   `product_category_name` column also shows `'unknown'` for those 13 rather than
   keeping the real PT name — a small loss of information.
3. **775 orders have no `order_items`** (a known Olist quirk — present in raw
   too). They are kept in `fct_orders` with `item_count = 0`; they simply have no
   rows in `fct_order_items`.
4. **Geography nulls pass through**: 269 customers / 7 sellers have null lat/lng
   because their zip prefix is absent from `geolocation`. Expected.


In [ ]:
# Star-schema diagram for dbt_dev_marts (fact constellation, 4 conformed dims).
# Pure matplotlib, no warehouse call. Boxes = tables, lines = key relationships.
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D

FACT   = "#2f6f9f"   # fact fill
DIM    = "#c9772e"   # dimension fill
CENTER = "#1f4e79"   # atomic fact fill

# name -> (x, y, w, h, kind, text)
boxes = {
    "dim_customer": (0.3, 7.7, 3.2, 1.9, "dim",
        "dim_customer  ·  96,096 rows\ngrain: customer_unique_id\nkey: customer_key (PK)\ncity/state, lat/lng, RFM inputs"),
    "dim_product": (6.4, 9.1, 3.2, 1.7, "dim",
        "dim_product  ·  32,951 rows\ngrain: product_id\nkey: product_key (PK)\ncategory PT/EN (unknown member)"),
    "dim_seller": (11.0, 7.2, 3.2, 1.7, "dim",
        "dim_seller  ·  3,095 rows\ngrain: seller_id\nkey: seller_key (PK)\ncity/state, lat/lng"),
    "dim_date": (5.05, 0.0, 4.4, 1.6, "dim",
        "dim_date  ·  852 rows  (conformed)\ngrain: 1 day, 2016-09-01 .. 2018-12-31\nkey: date_key (PK)  ·  is_complete_month"),
    "fct_order_items": (5.15, 4.7, 4.1, 2.0, "center",
        "fct_order_items  ·  112,650\nATOMIC SALES\ngrain: order_id + order_item_id\ndeg: order_id  ·  FK: customer / product / seller / date\nprice, freight_value, line_gross_value"),
    "fct_orders": (0.0, 3.5, 3.7, 1.9, "fact",
        "fct_orders  ·  99,441\ngrain: order_id  (accumulating snapshot)\nFK: customer_key, order_purchase_date\nlifecycle deltas, item + payment totals"),
    "fct_payments": (11.0, 4.0, 3.6, 1.7, "fact",
        "fct_payments  ·  103,886\ngrain: order_id + payment_sequential\ndeg: order_id  ·  payment_type, value"),
    "fct_reviews": (10.4, 1.4, 4.1, 1.7, "fact",
        "fct_reviews  ·  98,410  (deduped on review_id)\ngrain: 1 review  ·  deg: order_id\nreview_score, review_response_days"),
}

# (a, b, label, style):  'solid' = fact->dim (conformed FK),  'dashed' = fact->fact (shared order_id)
edges = [
    ("fct_order_items", "dim_customer", "customer_key", "solid"),
    ("fct_order_items", "dim_product",  "product_key",  "solid"),
    ("fct_order_items", "dim_seller",   "seller_key",   "solid"),
    ("fct_order_items", "dim_date",     "order_purchase_date", "solid"),
    ("fct_orders",      "dim_customer", "customer_key", "solid"),
    ("fct_orders",      "dim_date",     "order_purchase_date", "solid"),
    ("fct_reviews",     "dim_date",     "review_creation_date", "solid"),
    ("fct_order_items", "fct_orders",   "order_id", "dashed"),
    ("fct_payments",    "fct_orders",   "order_id", "dashed"),
    ("fct_reviews",     "fct_orders",   "order_id", "dashed"),
]

def center(name):
    x, y, w, h, *_ = boxes[name]
    return x + w / 2, y + h / 2

fig, ax = plt.subplots(figsize=(15, 11))

for a, b, label, style in edges:
    (x1, y1), (x2, y2) = center(a), center(b)
    ax.add_line(Line2D([x1, x2], [y1, y2], color="#8a8a8a",
                       lw=1.4, ls=style, zorder=1))
    t = 0.74 if style == "dashed" else 0.60   # push label toward the fact end
    lx, ly = x1 + (x2 - x1) * t, y1 + (y2 - y1) * t
    ax.text(lx, ly, label, fontsize=8, style="italic", color="#333",
            ha="center", va="center", zorder=5,
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.9))

for name, (x, y, w, h, kind, text) in boxes.items():
    fc = {"dim": DIM, "fact": FACT, "center": CENTER}[kind]
    ax.add_patch(FancyBboxPatch((x, y), w, h,
                                boxstyle="round,pad=0.02,rounding_size=0.12",
                                fc=fc, ec="black",
                                lw=2.0 if kind == "center" else 1.0,
                                alpha=0.97, zorder=3))
    ax.text(x + w / 2, y + h / 2, text, fontsize=8.4, color="white",
            ha="center", va="center", zorder=4,
            fontweight="bold" if kind == "center" else "normal")

legend = [
    Line2D([0], [0], marker="s", color="none", markerfacecolor=CENTER, markersize=15, label="atomic fact"),
    Line2D([0], [0], marker="s", color="none", markerfacecolor=FACT,   markersize=15, label="fact"),
    Line2D([0], [0], marker="s", color="none", markerfacecolor=DIM,    markersize=15, label="dimension"),
    Line2D([0], [0], color="#8a8a8a", lw=1.4, ls="solid",  label="fact → dimension  (conformed FK)"),
    Line2D([0], [0], color="#8a8a8a", lw=1.4, ls="dashed", label="fact → fact  (shared order_id)"),
]
ax.legend(handles=legend, loc="lower left", fontsize=9, framealpha=0.95)

ax.set_xlim(-0.2, 14.8)
ax.set_ylim(-0.4, 11.2)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("dbt_dev_marts — star / fact-constellation schema  (validated against olist_raw)",
             fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()


---

# Part 2 — Deep-dive analysis: three business questions

Built on the validated `dbt_dev_marts` star schema (Part 1). All three sections
read the marts layer through SQLAlchemy.

**Method — and why it keeps BigQuery cost low**

- Every heavy `GROUP BY` / `JOIN` runs **inside BigQuery**. The notebook only
  pulls back small aggregated results (tens to a few thousand rows).
- No `SELECT *` on a fact table — each query lists only the columns it needs.
- **pandas** (not polars) does the final shaping and charts: after in-warehouse
  aggregation the data is tiny, so pandas is the more readable, easier-to-edit
  choice. Charts are plain `matplotlib`. Only simple arithmetic is used
  (counts, %, averages, medians, cumulative sums) — no statistics or modelling.
- Everything in this part scans well under 1 GB in total.

**Data window.** Orders span **2016-09-04 → 2018-10-17**. Volume thins sharply
after **August 2018** (the source dump is partial past that point); reviews stop
at 2018-08-31. Time-trend charts are cut at the last complete month
(`dim_date.is_complete_month`).

**Data-quality exclusions used in this part** (the one-line filters are in the
setup cell):

| # | Rows | What | Why it is noise / an error | Handling |
|---|---|---|---|---|
| 1 | 2,965 orders | not delivered — `is_late IS NULL` (canceled, unavailable, in-transit) | "was it late?" is undefined with no delivery date | excluded from Section 1 only |
| 2 | 1,241 orders | `is_revenue_order = FALSE` (canceled / unavailable) | not realised revenue | excluded from every revenue figure |
| 3 | 775 orders | `item_price_total = 0` — order has no `order_items` (known Olist quirk, in raw too) | no sellable line ⇒ no product revenue | excluded from revenue figures (mostly overlaps #2) |
| 4 | 14 orders | `delivery_days > 180` (up to 209 d) — timestamp anomalies flagged in `int_order_lifecycle` | physically implausible transit; 0.015% of delivered orders | excluded from **average delivery-time** stats only; still counted as "late" |
| 5 | 825 customers | first order has **no** matching review | cannot be placed on the review scale; 1% of the cohort, behave differently (~23% repeat) | kept as their own "no review" row, never merged into 1–2★ |
| — | kept | 1 customer, R$13,664 lifetime revenue on a **single** order (RJ) | legitimate large one-off B2B-style purchase, not an error | kept — just not read as a "typical" customer |

City names in `dim_customer` are already lower-cased and accent-stripped
(4,298 distinct city/state pairs, **no** case/accent duplicates), so no
city-name cleaning is required.

In [ ]:
# --- Part 2 setup ---------------------------------------------------------------
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

# Project billed for the queries below. This is GCP_PROJECT from your .env; the
# warm-up cell near the top used a different project. Point this at whichever
# project you want the query cost charged to (both hold `dbt_dev_marts`).
BQ_PROJECT = "heroic-vial-505103-a3"
BQ_DATASET = "dbt_dev_marts"

engine = create_engine(f"bigquery://{BQ_PROJECT}/{BQ_DATASET}")

def q(sql: str) -> pd.DataFrame:
    """Run a query in BigQuery, return the (small, already-aggregated) result."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

BLUE, ORANGE, RED, GREEN = "#4c78a8", "#f58518", "#e45756", "#54a24b"
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True,
                     "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

# quick connection check
q("select count(*) as orders from fct_orders")


## Section 0 — Dataset overview (macro statistics)

A bird's-eye view before the three deep-dives: how big the dataset is, and the
headline distributions the use cases lean on — review scores for §1, repeat
behaviour for §2, geography for §3.

One caveat applies to everything below (see the exclusions table above): the
order flow is only **complete through August 2018**. Whole-span totals and rates
are fine; month-by-month figures past Aug-2018 are partial. Revenue figures use
realised orders only (`is_revenue_order`); average delivery time caps
`delivery_days` at 180 to drop the 14 timestamp-anomaly orders.

In [ ]:
# 0.1  Dataset at a glance — one row of headline numbers, shown transposed.
sql = """
select
    -- scale
    (select count(*) from fct_orders)                                             as orders_total,
    (select count(*) from fct_order_items)                                        as order_lines_total,
    (select count(*) from dim_customer)                                           as customers_total,
    (select count(distinct seller_key)  from fct_order_items)                     as sellers_with_sales,
    (select count(distinct product_key) from fct_order_items)                     as products_with_sales,
    (select count(distinct customer_state) from dim_customer)                     as customer_states,
    (select count(distinct concat(customer_city,'/',customer_state))
       from dim_customer)                                                         as customer_city_values,
    (select min(order_purchase_date) from fct_orders)                             as first_order_date,
    (select max(order_purchase_date) from fct_orders)                             as last_order_date,
    -- commercial (realised orders only)
    (select round(sum(item_price_total))    from fct_orders where is_revenue_order) as product_revenue_brl,
    (select round(sum(freight_value_total)) from fct_orders where is_revenue_order) as freight_brl,
    (select round(sum(item_gross_total))    from fct_orders where is_revenue_order) as gmv_incl_freight_brl,
    (select round(sum(payment_value))       from fct_payments)                      as payments_received_brl,
    (select countif(is_revenue_order) from fct_orders)                             as revenue_orders,
    (select round(100*avg(if(is_revenue_order,1,0)),1) from fct_orders)            as revenue_orders_pct,
    (select round(sum(item_price_total)/countif(is_revenue_order),2)
       from fct_orders where is_revenue_order)                                      as avg_order_value_brl,
    (select round(avg(item_count),2) from fct_orders where is_revenue_order)        as avg_items_per_order,
    -- customers
    (select round(100*avg(if(order_count>1,1,0)),2)          from dim_customer)    as repeat_rate_placed_pct,
    (select round(100*avg(if(delivered_order_count>1,1,0)),2) from dim_customer)   as repeat_rate_delivered_pct,
    (select round(avg(order_count),3)      from dim_customer)                      as avg_orders_per_customer,
    (select round(avg(lifetime_revenue),2) from dim_customer)                      as avg_lifetime_revenue_brl,
    (select round(approx_quantiles(lifetime_revenue,2)[offset(1)],2)
       from dim_customer)                                                          as median_lifetime_revenue_brl,
    -- delivery
    (select round(100*avg(if(order_delivered_customer_date is not null,1,0)),1)
       from fct_orders)                                                            as pct_orders_delivered,
    (select round(100*avg(if(is_late,1,0)),1)
       from fct_orders where is_late is not null)                                  as late_rate_of_delivered_pct,
    (select round(avg(delivery_days),1)
       from fct_orders where delivery_days between 0 and 180)                      as avg_delivery_days,
    (select round(approx_quantiles(delivery_days,2)[offset(1)],1)
       from fct_orders where delivery_days between 0 and 180)                      as median_delivery_days,
    (select round(avg(date_diff(date(order_delivered_customer_date),
                                date(order_estimated_delivery_date), day)),1)
       from fct_orders where is_late is not null)                                  as avg_days_late_vs_promise,
    -- reviews  (negative avg_days_late_vs_promise = usually delivered early)
    (select round(100 * (select count(distinct order_id) from fct_reviews)
                       / (select count(*) from fct_orders), 1))                    as pct_orders_with_review,
    (select round(avg(review_score),2)  from fct_reviews)                          as avg_review_score,
    (select round(100*avg(if(review_score=5,1,0)),1)  from fct_reviews)            as pct_5_star,
    (select round(100*avg(if(review_score<=2,1,0)),1) from fct_reviews)            as pct_1_2_star,
    (select round(avg(review_response_days),1) from fct_reviews)                   as avg_review_response_days,
    -- payments
    (select round(avg(payment_installments),2) from fct_payments)                 as avg_installments,
    (select round(100*avg(if(payment_installments>1,1,0)),1) from fct_payments)   as pct_paid_in_installments
"""
glance = q(sql).T.rename(columns={0: "value"})
display(glance)


In [ ]:
# 0.2  Distribution (percentiles) of the key continuous measures.
#      Means are in 0.1; this shows spread and skew. cast(... as float64) keeps
#      every branch of the UNION the same array type.
sql = """
with quant as (
    select 'order product value (R$)' as measure,
           approx_quantiles(cast(item_price_total as float64), 100) as p
      from fct_orders where is_revenue_order
    union all
    select 'freight per order (R$)',
           approx_quantiles(cast(freight_value_total as float64), 100)
      from fct_orders where is_revenue_order
    union all
    select 'items per order',
           approx_quantiles(cast(item_count as float64), 100)
      from fct_orders where is_revenue_order
    union all
    select 'delivery days (capped 0-180)',
           approx_quantiles(cast(delivery_days as float64), 100)
      from fct_orders where delivery_days between 0 and 180
    union all
    select 'review score (1-5)',
           approx_quantiles(cast(review_score as float64), 100)
      from fct_reviews
    union all
    select 'lifetime revenue per customer (R$)',
           approx_quantiles(cast(lifetime_revenue as float64), 100)
      from dim_customer
)
select measure,
       round(p[offset(0)],   2) as min,
       round(p[offset(10)],  2) as p10,
       round(p[offset(25)],  2) as p25,
       round(p[offset(50)],  2) as median,
       round(p[offset(75)],  2) as p75,
       round(p[offset(90)],  2) as p90,
       round(p[offset(99)],  2) as p99,
       round(p[offset(100)], 2) as max
from quant
"""
display(q(sql))


In [ ]:
# 0.3  Review-score distribution for the whole dataset.
sql = """
select review_score,
       count(*)                                         as reviews,
       round(100 * count(*) / sum(count(*)) over (), 1) as pct
from fct_reviews
group by review_score
order by review_score
"""
rev = q(sql)
avg_score = (rev["review_score"] * rev["reviews"]).sum() / rev["reviews"].sum()
display(rev)
print(f"overall average review score: {avg_score:.2f}    "
      f"(1-2 star {rev.loc[rev.review_score <= 2, 'pct'].sum():.1f}%   "
      f"5 star {rev.loc[rev.review_score == 5, 'pct'].iat[0]:.1f}%)")

ax = rev.plot(x="review_score", y="pct", kind="bar", color=BLUE, legend=False)
ax.set_xlabel("review score"); ax.set_ylabel("% of reviews")
ax.set_title(f"Review scores are bimodal — mean {avg_score:.2f}, but mostly 5s and 1s")
for i, v in enumerate(rev["pct"]):
    ax.text(i, v + 0.6, f"{v:.1f}%", ha="center")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


In [ ]:
# 0.4  Order-status breakdown — the funnel, and what §1 leaves out.
sql = """
select order_status,
       count(*)                                         as orders,
       round(100 * count(*) / sum(count(*)) over (), 2) as pct_orders,
       round(sum(item_price_total), 0)                  as product_revenue
from fct_orders
group by order_status
order by orders desc
"""
display(q(sql))


In [ ]:
# 0.5  Monthly product revenue and order volume (realised orders).
#      Faded bars = incomplete months - do NOT read them as a decline.
sql = """
select d.year_month,
       d.is_complete_month,
       count(*)                          as orders,
       round(sum(f.item_price_total), 0) as product_revenue
from fct_orders f
join dim_date d on d.date_key = f.order_purchase_date
where f.is_revenue_order
  and d.year_month >= '2016-10'
group by d.year_month, d.is_complete_month
order by d.year_month
"""
mth = q(sql)
bar_colours = [BLUE if c else "#c9d6e3" for c in mth["is_complete_month"]]

fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.bar(mth["year_month"], mth["product_revenue"], color=bar_colours)
ax1.set_ylabel("product revenue (R$)")
ax1.tick_params(axis="x", rotation=90)
ax2 = ax1.twinx()
ax2.plot(mth["year_month"], mth["orders"], color=RED, marker="o", markersize=3)
ax2.set_ylabel("orders"); ax2.grid(False)
ax1.set_title("Monthly product revenue & orders — steady growth; faded bars = incomplete month")
plt.tight_layout(); plt.show()


In [ ]:
# 0.6  Payment-method mix.
sql = """
select payment_type,
       count(*)                                         as payments,
       round(100 * count(*) / sum(count(*)) over (), 1) as pct_of_payments,
       round(sum(payment_value), 0)                     as total_value,
       round(avg(payment_installments), 1)              as avg_installments
from fct_payments
group by payment_type
order by payments desc
"""
display(q(sql))


### Section 0 — what the macro picture says

**Scale.** ~99.4k orders from ~96.1k unique customers, ~3.1k sellers, ~33k
products, **Sep 2016 -> Oct 2018** (~24 usable months). **R$13.5M** product
revenue (R$15.7M with freight; R$16.0M actually paid). 98.8% of orders count as
revenue. The median order is **R$87** (mean R$137 - right-skewed) with **1.14
items** - a single-item, mid-ticket marketplace.

**For Section 1 - delivery -> reviews**
- **Mean review score 4.09**, but the distribution is bimodal: **57.8% five-star,
  ~14.6% one-or-two-star, 8.2% three-star** (cell 0.3). The average barely moves;
  the *detractor share* is the sensitive metric - which is why Section 1 tracks it.
- **97% of orders are delivered**, and of those **only 8.1% are late**. On
  average orders arrive **~12 days *before* the promised date**
  (`avg_days_late_vs_promise = -11.9`): Olist's delivery estimates are heavily
  padded, which is *why* "on time" dominates. The 8% that miss even a padded
  promise are real failures.
- Median delivery time **10 days** (mean 12.5, p90 23, p99 46). Reviews are
  answered in ~3 days and cover **98.6%** of orders, so Section 1's review-delivery
  join loses almost nothing.

**For Section 2 - retention**
- **Repeat rate 3.1%** (placed 2+) / 2.9% (delivered 2+); **1.035 orders per
  customer**; ~97% one-and-done. That is the Section 2 problem statement in one line.
- Customer value is skewed: **mean lifetime revenue R$164, median R$107**, p90
  R$317, max R$13,664 - a thin tail of high-value buyers.

**For Section 3 - geography**
- **27 states, ~4,300 distinct city/state values.** "City" is a high-cardinality
  raw string with a long thin tail; Section 3 quantifies the concentration.

**Context.** Payments are **74% credit card** (avg **3.5 installments**), 19%
boleto, 6% voucher; ~49% of payment records are instalment plans - normal for
Brazil.

**Anomalies these tables surface** (all consistent with the Part 2 exclusions):

- **`order_status` (cell 0.4):** 96,478 `delivered`, then `shipped` 1,107 /
  `canceled` 625 / `unavailable` 609 / `invoiced` 314 / `processing` 301 /
  `created` 5 / `approved` 2. The last few never reached the customer;
  `shipped` / `invoiced` / `processing` still count as revenue (the project's
  `revenue_order_statuses`), which is why whole-dataset product revenue here
  (**R$13.49M**, revenue orders) sits just above Section 1's figure (**R$13.22M**,
  *delivered* orders only) - same data, narrower scope.
- **Right-skew, not errors:** `items per order` max **21** (p99 = 3), order value
  max **R$13,440**, lifetime revenue max **R$13,664**. Real high-end orders that
  pull every mean above its median - use medians for "typical".
- **Zeros:** `min` order value / freight / items = **0** - the 775 item-less
  orders (already dropped from revenue by `is_revenue_order`).
- **`delivery days` max shows ~175** only because cells 0.1/0.2 cap at 180; 14
  orders run longer (up to 209 d) - the timestamp anomalies flagged earlier.
- **`payment_type = 'not_defined'`** on 3 records with R$0 - a source
  placeholder, immaterial.


## Section 1 — Does late delivery drag down reviews, and does that hit revenue?

**Questions.** (a) How far do review scores fall as an order runs late?
(b) Does a poor first delivery reduce what a customer is worth afterwards?
(c) How much revenue is exposed to late delivery?

**Key metrics.** average review score and **detractor rate** (share scoring 1–2)
by lateness bucket; repeat-purchase rate and average lifetime revenue split by
the customer's **first** delivery experience; share of realised product revenue
that passed through a late delivery.

Scope: delivered orders that count as revenue and carry a review.

In [ ]:
# 1.1  Review score vs how late the delivery was.
#      days_late = actual delivered date - promised date  (negative = early).
sql = """
with o as (
    select order_id, is_late,
           date_diff(date(order_delivered_customer_date),
                     date(order_estimated_delivery_date), day) as days_late,
           item_price_total
    from fct_orders
    where is_late is not null        -- delivered orders only
      and is_revenue_order           -- exclude canceled / unavailable
),
r as (select order_id, review_score from fct_reviews)
select
    case when not o.is_late     then '0  on time / early'
         when o.days_late <=  3 then '1  1-3 days late'
         when o.days_late <=  7 then '2  4-7 days late'
         when o.days_late <= 15 then '3  8-15 days late'
         else                        '4  16+ days late'
    end                                                 as delivery_bucket,
    count(*)                                             as orders,
    round(avg(r.review_score), 2)                        as avg_review_score,
    round(100 * avg(if(r.review_score <= 2, 1, 0)), 1)   as detractor_rate_pct,
    round(100 * avg(if(r.review_score  = 5, 1, 0)), 1)   as five_star_pct
from o join r using (order_id)
group by delivery_bucket
order by delivery_bucket
"""
d1 = q(sql)
display(d1)

fig, ax = plt.subplots()
ax.bar(d1["delivery_bucket"], d1["avg_review_score"], color=BLUE)
ax.set_ylim(0, 5); ax.set_ylabel("avg review score (1-5)")
for i, v in enumerate(d1["avg_review_score"]):
    ax.text(i, v + 0.12, f"{v:.2f}", ha="center")
ax2 = ax.twinx()
ax2.plot(d1["delivery_bucket"], d1["detractor_rate_pct"], color=RED, marker="o")
ax2.set_ylim(0, 100); ax2.set_ylabel("detractor rate %  (1-2 stars)"); ax2.grid(False)
ax.set_title("Review score collapses once a delivery is more than 3 days late")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()


In [ ]:
# 1.2  Does the FIRST delivery experience change what the customer is worth later?
#      Cohort: first order placed on/before 2018-06-30 -> everyone has had at
#      least ~3.5 months of room to come back (fair comparison).
sql = """
with ranked as (
    select customer_key, order_id, order_purchase_date, is_late,
           row_number() over (partition by customer_key
                              order by order_purchase_timestamp, order_id) as rn
    from fct_orders
),
first_order as (
    select customer_key, order_id, is_late
    from ranked
    where rn = 1 and order_purchase_date <= date '2018-06-30'
),
cust as (select customer_key, order_count, lifetime_revenue from dim_customer)
select
    case when f.is_late is null then 'undelivered first order'
         when f.is_late         then 'first order LATE'
         else                        'first order on time'
    end                                               as first_experience,
    count(*)                                          as customers,
    round(100 * avg(if(c.order_count > 1, 1, 0)), 2)  as repeat_rate_pct,
    round(avg(c.lifetime_revenue), 2)                 as avg_lifetime_revenue
from first_order f
left join cust c using (customer_key)
group by first_experience
order by first_experience
"""
d2 = q(sql)
display(d2)

ax = d2.plot(x="first_experience", y="repeat_rate_pct", kind="bar",
             color=BLUE, legend=False)
ax.set_ylabel("repeat rate %"); ax.set_title("Repeat rate by first-delivery experience")
for i, v in enumerate(d2["repeat_rate_pct"]):
    ax.text(i, v + 0.05, f"{v:.2f}%", ha="center")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


In [ ]:
# 1.3  Is late delivery getting better or worse? (complete months only)
sql = """
select d.year_month,
       count(*)                                  as delivered_orders,
       round(100 * avg(if(f.is_late, 1, 0)), 1)  as late_rate_pct
from fct_orders f
join dim_date d on d.date_key = f.order_purchase_date
where f.is_late is not null
  and f.is_revenue_order
  and d.is_complete_month
  and d.date_key >= date '2016-10-01'   -- drop 2 near-empty months at the start
group by d.year_month
order by d.year_month
"""
d3 = q(sql)

plt.plot(d3["year_month"], d3["late_rate_pct"], marker="o", color=RED)
plt.axhline(d3["late_rate_pct"].mean(), ls="--", color="grey",
            label=f"mean {d3['late_rate_pct'].mean():.1f}%")
plt.ylabel("% of delivered orders that were late")
plt.title("Late-delivery rate by month — spikes line up with peak demand")
plt.xticks(rotation=90); plt.legend(); plt.tight_layout(); plt.show()

print("Worst months:")
display(d3[d3["late_rate_pct"] >= 10])


In [ ]:
# 1.4  How much realised product revenue passed through a late delivery?
sql = """
select
    count(*)                                             as delivered_revenue_orders,
    round(100 * avg(if(is_late, 1, 0)), 2)               as late_order_rate_pct,
    round(sum(item_price_total), 0)                      as product_revenue,
    round(sum(if(is_late, item_price_total, 0)), 0)      as revenue_via_late_delivery,
    round(100 * sum(if(is_late, item_price_total, 0))
              / sum(item_price_total), 2)                as pct_revenue_via_late
from fct_orders
where is_late is not null and is_revenue_order
"""
display(q(sql).T.rename(columns={0: "value"}))


### Section 1 — findings

**1. Reviews collapse the moment a delivery slips past its promised date, and
almost all the damage is done inside the first week late.**

| First delivery outcome | Orders | Avg review | Detractors (1–2★) | 5★ |
|---|--:|--:|--:|--:|
| On time / early | 87,963 | **4.30** | 9.2% | 62.5% |
| 1–3 days late | 3,117 | 3.60 | 23.9% | 40.4% |
| 4–7 days late | 1,742 | **2.10** | 67.6% | 14.1% |
| 8–15 days late | 1,597 | 1.67 | 80.0% | 6.7% |
| 16+ days late | 1,177 | 1.72 | 78.4% | 7.3% |

A delivery only 4–7 days late already turns the majority of buyers into
detractors (68%). Past ~8 days the score is floored near 1.7 — being *very* late
is no worse than being *moderately* late.

**2. ~R$1.16M of realised product revenue (8.8%) flows through late deliveries**,
on ~8% of delivered orders (see cell 1.4). That revenue is booked; the exposure
is reputational and future demand.

**3. The downstream repeat effect is real but small here** (cell 1.2, first-order
cohort placed on/before 2018-06-30):

| First delivery | Customers | Repeat rate | Avg lifetime revenue |
|---|--:|--:|--:|
| On time | 74,496 | **3.41%** | R$164 |
| Late | 6,681 | **2.78%** | R$180 |
| Undelivered | 2,571 | 4.36% | R$120 |

On-time lifts the repeat rate by ~0.6pp (≈ +23% relative). Because the base
repeat rate is only ~3% and the window is short, that gap is worth only ~R$5k of
second-order revenue across this period — the reason to fix late delivery is the
review / reputation hit on a marketplace, not measurable near-term churn.
Detractors are **not** low-value: their orders are slightly larger and heavier,
which is *why* they shipped late. (The "undelivered first order" group re-orders
*more* — a missing parcel usually gets re-purchased or the order is remade.)

**4. Lateness is a peak-season capacity problem.** Baseline ~4–6%, but
Nov-2017 (Black Friday) 14%, Feb-2018 16%, **Mar-2018 21%**, Aug-2018 10%.
Padding promised dates and pre-booking carrier capacity for those windows is
where the review damage concentrates.

**Answer.** Late delivery drives a steep, well-defined drop in review scores.
Its revenue impact is mostly indirect: ~R$1.2M of revenue and the platform's
review reputation ride on the ~8% of orders that arrive late, clustered in a
handful of high-demand months.

## Section 2 — Low repeat-purchase rate: why, and what to do about it

Diagnosis first, then the strategy. Metrics: order-count distribution and
revenue contribution by segment; time between 1st and 2nd order; repeat rate cut
by first-purchase category and state; size of the lapsed base available for
win-back. (First-delivery experience was already covered in Section 1.)

In [ ]:
# 2.1  How concentrated is the base on one-time buyers, and how much revenue do
#      repeat buyers actually contribute?
sql = """
with c as (select order_count, lifetime_revenue from dim_customer)
select
    case when order_count = 1 then '1 order'
         when order_count = 2 then '2 orders'
         when order_count = 3 then '3 orders'
         else                      '4+ orders'
    end                                                      as segment,
    count(*)                                                 as customers,
    round(100 * count(*) / sum(count(*)) over (), 2)         as pct_customers,
    round(sum(lifetime_revenue), 0)                          as revenue,
    round(100 * sum(lifetime_revenue)
              / sum(sum(lifetime_revenue)) over (), 2)       as pct_revenue,
    round(avg(lifetime_revenue), 2)                          as avg_lifetime_revenue
from c
group by segment
order by segment
"""
d5 = q(sql)
display(d5)

ax = d5.plot(x="segment", y=["pct_customers", "pct_revenue"], kind="bar",
             color=[BLUE, ORANGE])
ax.set_ylabel("% of total"); plt.xticks(rotation=0)
ax.set_title("96.9% of customers order once (94% of revenue); "
             "repeat buyers are worth ~2x each")
plt.tight_layout(); plt.show()


In [ ]:
# 2.2  For customers who DID buy again, how long until the second order?
#      Tells us WHEN a retention nudge should land.
sql = """
with ranked as (
    select customer_key, order_purchase_date,
           row_number() over (partition by customer_key
                              order by order_purchase_timestamp, order_id) as rn
    from fct_orders
)
select date_diff(b.order_purchase_date, a.order_purchase_date, day) as gap_days
from ranked a
join ranked b using (customer_key)
where a.rn = 1 and b.rn = 2
"""
gaps = q(sql)["gap_days"]
print(f"repeat customers      : {len(gaps):,}")
print(f"median gap            : {gaps.median():.0f} days")
print(f"25th / 75th percentile: {gaps.quantile(.25):.0f} / {gaps.quantile(.75):.0f} days")
print(f"2nd order within 30/60/90 days: "
      f"{(gaps <= 30).mean()*100:.0f}% / {(gaps <= 60).mean()*100:.0f}% / {(gaps <= 90).mean()*100:.0f}%")

plt.hist(gaps.clip(upper=365), bins=37, color=BLUE)
plt.axvline(gaps.median(), color=RED, ls="--", label=f"median {gaps.median():.0f} days")
plt.xlabel("days between 1st and 2nd order  (capped at 365)")
plt.ylabel("customers")
plt.title("Half of all second orders happen within ~4 weeks of the first")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# 2.3  Which first-purchase category produces repeat customers, and which are
#      "one and done"?  Category = the highest-priced line in the first order.
sql = """
with ranked as (
    select customer_key, order_id, order_purchase_date,
           row_number() over (partition by customer_key
                              order by order_purchase_timestamp, order_id) as rn
    from fct_orders
),
first_order as (
    select customer_key, order_id from ranked
    where rn = 1 and order_purchase_date <= date '2018-06-30'
),
first_line as (
    select fi.order_id, dp.product_category_name_english as category,
           row_number() over (partition by fi.order_id
                              order by fi.price desc, fi.order_item_id) as pr
    from fct_order_items fi
    join dim_product dp on dp.product_key = fi.product_key
),
cust as (select customer_key, order_count from dim_customer)
select l.category,
       count(*)                                         as customers,
       round(100 * avg(if(c.order_count > 1, 1, 0)), 2) as repeat_rate_pct
from first_order f
join first_line l on l.order_id = f.order_id and l.pr = 1
join cust c using (customer_key)
group by l.category
having customers >= 500
order by repeat_rate_pct desc
"""
d6 = q(sql)
overall = 100 * d5.loc[d5["segment"] != "1 order", "customers"].sum() / d5["customers"].sum()

fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4))
for a, part, colour, ttl in [(axa, d6.head(6), GREEN, "Highest repeat rate"),
                             (axb, d6.tail(6), RED,   "Lowest repeat rate")]:
    part = part.iloc[::-1]
    a.barh(part["category"], part["repeat_rate_pct"], color=colour)
    a.axvline(overall, color="grey", ls="--")
    a.set_xlabel("repeat rate %"); a.set_title(f"{ttl} by first category")
axa.text(overall, -0.6, f" overall {overall:.1f}%", color="grey", fontsize=8)
plt.tight_layout(); plt.show()
display(d6)


In [ ]:
# 2.4  Repeat rate and average customer value by state (>= 1000 customers).
sql = """
select customer_state as state,
       count(*)                                        as customers,
       round(100 * avg(if(order_count > 1, 1, 0)), 2)  as repeat_rate_pct,
       round(avg(lifetime_revenue), 2)                 as avg_lifetime_revenue
from dim_customer
group by state
having customers >= 1000
order by customers desc
"""
d7 = q(sql)
display(d7)
print("Repeat rate barely moves across states "
      f"({d7['repeat_rate_pct'].min():.1f}%-{d7['repeat_rate_pct'].max():.1f}%) "
      "-> geography is not a retention lever.")


In [ ]:
# 2.5  How large is the lapsed base for a win-back campaign?
#      NOTE: recency_days is measured from a fixed reference (~end of 2018), NOT
#      "today", because the data is historical. Read these as relative sizes.
sql = """
select
    case when recency_days <= 180 then '0-180 days'
         when recency_days <= 365 then '181-365 days'
         when recency_days <= 540 then '366-540 days'
         else                          '540+ days'
    end                                              as time_since_last_order,
    count(*)                                         as customers,
    round(100 * avg(if(order_count > 1, 1, 0)), 2)   as repeat_rate_pct,
    round(sum(lifetime_revenue), 0)                  as revenue_to_date
from dim_customer
group by time_since_last_order
order by time_since_last_order
"""
display(q(sql))


### Section 2 — diagnosis and retention strategy

**Diagnosis**

- **96.9%** of customers (93,099) order exactly once; only **3.1%** place a
  second order, **0.26%** a third.
- One-time buyers are **94%** of revenue — the base is healthy — but a 2-order
  customer is worth **R$288 vs R$159** (1.8×), a 3-order customer R$428. Repeat
  is a pure margin lever.
- **When** repeat happens: median **28 days** after the first order; **51%** of
  all second orders land within 30 days, **69%** within 90. The retention
  window is the *first month*.
- **What** retains: first-purchase category matters — home appliances (8.9%),
  fashion bags/accessories (6.5%), furniture & décor (5.0%), bed/bath/table
  (4.9%) breed repeat buyers; electronics (1.8%), "cool stuff" (1.9%),
  consoles/games (1.9%), watches/gifts (2.4%) are one-and-done gift / big-ticket
  categories.
- **Geography** is not a lever: repeat rate is a flat 2.2–3.4% across every
  large state (slightly worse in the Northeast, where delivery is slower — links
  back to Section 1).
- **Lapsed pool**: ≈ 43k customers last ordered 366+ days ago and another ≈ 41k
  at 181–365 days (relative to the dataset's fixed reference date) — a large
  win-back target.

**Strategy — five plays, each tied to a number above**

| # | Play | Target segment | Trigger / timing | Why the data supports it |
|---|---|---|---|---|
| 1 | **"Second order" lifecycle nudge** — reminder + category-relevant picks + small time-boxed voucher | every first-time buyer, order delivered & rated 4–5★ | fire **day 20–25**, just before the 28-day median gap; expire day 45 | 51% of repeats already happen ≤30 days — capture demand that exists instead of losing it |
| 2 | **Category cross-sell** — move one-time buyers from a low-repeat category to an adjacent high-repeat one (electronics → computer/home accessories; watches/gifts → fashion accessories) | one-time buyers whose first category repeats < 3% | 30–60 days after delivery | low-repeat categories are one-off purchases; the customer isn't bad, the *category* doesn't recur |
| 3 | **Service recovery** — proactive apology + credit for any late delivery, *before* the review request | orders flagged `is_late` | within 24–48h of delivery | 68% of 4–7-day-late buyers become detractors; a credit pre-empts the 1–2★ review and gives a reason to reorder |
| 4 | **Delivery-promise padding at peak** — widen estimated dates and pre-book carrier capacity for Nov and Feb–Mar | operations | seasonal | late rate hits 14–21% those months vs ~5% baseline; most review damage is there |
| 5 | **Win-back campaign** — staggered offers to lapsed buyers; best offer to those whose single order was high-value or 5★ | 181–540 days since last order (~68k customers) | one-off, then quarterly | large dormant pool; a good first experience makes reactivation cheap |

**Expected impact** (simple arithmetic — see cell 2.6): lifting repeat from
**3.1% → 5%** ≈ **+1,800 repeat customers** × **~R$128** incremental value of a
second order ≈ **R$232k** additional revenue; → 6% ≈ R$356k. Order-of-magnitude,
not a forecast.

**How to measure.** Headline KPI = repeat rate at **30 / 60 / 90 days after
first delivery**, split by the play that touched the customer, each with a
holdout. Guardrail = detractor rate on late orders (Section 1).

In [ ]:
# 2.6  Rough size of the prize: what is a few points of repeat rate worth?
#      Plain arithmetic, all inputs pulled from d5 (cell 2.1) so edits flow through.
one_order   = d5.loc[d5["segment"] == "1 order",  "avg_lifetime_revenue"].iat[0]
two_orders  = d5.loc[d5["segment"] == "2 orders", "avg_lifetime_revenue"].iat[0]
incremental = two_orders - one_order                      # value of getting a 2nd order
customers   = int(d5["customers"].sum())
repeat_now  = d5.loc[d5["segment"] != "1 order", "customers"].sum() / customers

print(f"customers                     : {customers:,}")
print(f"repeat rate now               : {repeat_now*100:.1f}%")
print(f"incremental value of 2nd order : R$ {incremental:,.2f}\n")
for target in (0.04, 0.05, 0.06):
    extra = customers * (target - repeat_now)
    print(f"  repeat {repeat_now*100:.1f}% -> {target*100:.0f}%  "
          f"= +{extra:,.0f} repeat customers  ~ R$ {extra*incremental:,.0f} revenue")


## Section 3 — How is revenue distributed across cities?

Revenue = product price (`item_price_total`), realised orders only
(`is_revenue_order`). Customers are located by
`dim_customer.customer_city / customer_state` — the customer's city (already
lower-cased and accent-stripped in the mart), i.e. **demand** geography, not
seller/fulfilment origin. Freight is excluded; adding it lifts every city's
figure by roughly a proportional ~15%.

In [ ]:
# 3.1  Revenue by customer state.
sql = """
with o as (
    select customer_key, item_price_total
    from fct_orders
    where is_revenue_order
)
select d.customer_state                   as state,
       count(*)                           as orders,
       count(distinct o.customer_key)     as customers,
       round(sum(o.item_price_total), 0)  as revenue,
       round(100 * sum(o.item_price_total)
                 / sum(sum(o.item_price_total)) over (), 2) as pct_revenue,
       round(sum(o.item_price_total) / count(*), 2)         as avg_order_value
from o
join dim_customer d using (customer_key)
group by state
order by revenue desc
"""
states = q(sql)
states["cum_pct_revenue"] = states["pct_revenue"].cumsum().round(1)
display(states.head(12))

ax = states.head(12).plot(x="state", y="pct_revenue", kind="bar", color=BLUE, legend=False)
ax.set_ylabel("% of revenue"); plt.xticks(rotation=0)
ax.set_title(f"Sao Paulo state = {states.iloc[0]['pct_revenue']:.0f}% of revenue; "
             f"top 3 states = {states['pct_revenue'].head(3).sum():.0f}%")
plt.tight_layout(); plt.show()


In [ ]:
# 3.2  Top cities by revenue, with cumulative share (Pareto view).
sql = """
with o as (
    select customer_key, item_price_total
    from fct_orders
    where is_revenue_order
)
select concat(d.customer_city, ' / ', d.customer_state) as city,
       count(*)                              as orders,
       count(distinct o.customer_key)        as customers,
       round(sum(o.item_price_total), 0)     as revenue,
       round(100 * sum(o.item_price_total)
                 / sum(sum(o.item_price_total)) over (), 2) as pct_revenue
from o
join dim_customer d using (customer_key)
group by city
order by revenue desc
limit 20
"""
cities = q(sql)
cities["cum_pct_revenue"] = cities["pct_revenue"].cumsum().round(1)
display(cities)

fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.bar(cities["city"], cities["revenue"], color=BLUE)
ax1.set_ylabel("revenue (R$)"); ax1.tick_params(axis="x", rotation=90)
ax2 = ax1.twinx()
ax2.plot(cities["city"], cities["cum_pct_revenue"], color=RED, marker="o")
ax2.set_ylabel("cumulative % of ALL revenue"); ax2.set_ylim(0, 50); ax2.grid(False)
ax1.set_title("Top 20 cities carry ~40% of revenue; the rest is ~4,300 small cities")
plt.tight_layout(); plt.show()


In [ ]:
# 3.3  Concentration and the long tail (aggregate every city once, then pandas).
sql = """
with o as (select customer_key, item_price_total from fct_orders where is_revenue_order)
select concat(d.customer_city, ' / ', d.customer_state) as city,
       count(*)                          as orders,
       round(sum(o.item_price_total), 2) as revenue
from o
join dim_customer d using (customer_key)
group by city
"""
allc = q(sql).sort_values("revenue", ascending=False).reset_index(drop=True)
total = allc["revenue"].sum()

for n in (1, 5, 10, 25, 50, 100):
    print(f"top {n:>3} cities : {allc['revenue'].head(n).sum()/total*100:5.1f}% of revenue")
tail = allc["orders"] <= 5
print(f"\ntotal city/state values : {len(allc):,}")
print(f"with exactly 1 order    : {(allc['orders'] == 1).sum():,}")
print(f"with <= 5 orders        : {tail.sum():,}  "
      f"(together {allc.loc[tail, 'revenue'].sum()/total*100:.1f}% of revenue)")


### Section 3 — findings

- **São Paulo state alone is 38% of revenue**; RJ 13%, MG 12%. **Top 3 states =
  63%**; top 7 (SP, RJ, MG, RS, PR, SC, BA) = **82%**.
- By city: **São Paulo 14.1%**, Rio de Janeiro 7.3%, Belo Horizonte 2.6%,
  Brasília 2.2%. **Top 10 cities = 34%**, top 50 = 52%, top 100 = 61% of all
  revenue.
- The remaining ~39% is spread over **~4,200 smaller cities**. **1,264
  city/state values have exactly one order**, and 2,905 have ≤5 — individually
  meaningless, but together only **~7%** of revenue, so nothing material is lost
  by analysing them in aggregate.
- **Average order value runs opposite to volume**: lowest in SP (R$126), highest
  in the distant Northeast — CE R$171, PE R$159, BA R$152. Far customers order
  less often but bigger.

**Anomaly / data-quality notes**

- The long tail is **not an error**, but don't read it city-by-city: 2,905
  cities with ≤5 orders carry too little signal for per-city conclusions. Cell
  3.3 ranks every city once and treats everything below the top ~20–50 as an
  aggregate. Nothing is deleted.
- `"quilometro 14 do mutum / ES"` looks odd but is a real rural locality
  (Km 14, Mutum) — kept.
- Figures exclude the 1,241 non-revenue and 775 item-less orders (see the
  exclusions table at the top of Part 2).

**Answer.** Revenue is **highly concentrated** — about two-thirds from three
states and a third from ten cities, with São Paulo dominating both. Growth
planning splits cleanly: defend the SP / RJ / MG core, and work the long tail at
**region** level, not city level.